In [1]:
!pip install -q langchain-typesafe

In [15]:
!pip install -q requests

import requests
import os
from google.colab import userdata

os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

print("Jev API key loaded:", bool(os.environ.get("TYPESAFE_API_KEY")))

Jev API key loaded: True


In [3]:
from langchain_typesafe import TypeSafeClassifier, Choice, Score, Noul

In [4]:
classifier = TypeSafeClassifier()

/tmp/ipykernel_2194/4292118559.py:1: LangChainBetaWarning: The class `TypeSafeClassifier` is in beta. It is actively being worked on, so the API may change.
  classifier = TypeSafeClassifier()


In [5]:
review = """
The movie was amazing! The acting was excellent,
the story was interesting, and I really enjoyed it.
"""

In [9]:
import os

key = os.environ.get("TYPESAFE_API_KEY")

print("Key exists:", key is not None)
print("Key length:", len(key) if key else 0)

Key exists: True
Key length: 56


In [12]:
payload = {
    "model": "jev-1.13",

    "state": review,

    "questions": {

        "positive": {
            "type": "noul",
            "instructions": "Is this a positive movie review?"
        },

        "category": {
            "type": "choice",
            "instructions": "What type of review is this?",
            "criteria": {
                "positive": "The reviewer clearly likes the movie.",
                "negative": "The reviewer clearly dislikes the movie.",
                "mixed": "The reviewer has both positive and negative opinions."
            }
        },

        "strength": {
            "type": "score",
            "instructions": "How strong is the positive sentiment?",
            "criteria": [
                "Weak or slightly positive",
                "Moderately positive",
                "Extremely positive"
            ]
        }
    }
}

In [17]:
response = requests.post(
    "https://jev-ai.org/api/v1/systemone/",
    headers={
        "Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}",
        "Content-Type": "application/json"
    },
    json=payload
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"id":"dec_1d2ee91768f1b197ff51730f","model":"jev-1.13","model_version":"jev-1.13-20260917","provider":"jev-ai.org","answers":{"positive":{"type":"noul","noul":0.99},"category":{"type":"choice","choice":"positive","probabilities":{"positive":1,"mixed":0,"negative":0},"confidence":1},"strength":{"type":"score","score":2,"legend":{"0":"Weak or slightly positive","1":"Moderately positive","2":"Extremely positive"},"probabilities":{"0":0,"1":0,"2":1},"confidence":1}},"usage":{"input_tokens":420,"output_tokens":68,"charged_tokens":0,"charged_credits":1,"wallet":"credits"},"latency_ms":200}


In [18]:
result = response.json()

answers = result["answers"]

print(answers)

{'positive': {'type': 'noul', 'noul': 0.99}, 'category': {'type': 'choice', 'choice': 'positive', 'probabilities': {'positive': 1, 'mixed': 0, 'negative': 0}, 'confidence': 1}, 'strength': {'type': 'score', 'score': 2, 'legend': {'0': 'Weak or slightly positive', '1': 'Moderately positive', '2': 'Extremely positive'}, 'probabilities': {'0': 0, '1': 0, '2': 1}, 'confidence': 1}}


In [19]:
print("Positive:", answers["positive"])
print("Category:", answers["category"])
print("Strength:", answers["strength"])

Positive: {'type': 'noul', 'noul': 0.99}
Category: {'type': 'choice', 'choice': 'positive', 'probabilities': {'positive': 1, 'mixed': 0, 'negative': 0}, 'confidence': 1}
Strength: {'type': 'score', 'score': 2, 'legend': {'0': 'Weak or slightly positive', '1': 'Moderately positive', '2': 'Extremely positive'}, 'probabilities': {'0': 0, '1': 0, '2': 1}, 'confidence': 1}


In [20]:
positive_probability = answers["positive"]["noul"]

category = answers["category"]["choice"]

strength = answers["strength"]["score"]

print("Positive probability:", positive_probability)
print("Category:", category)
print("Strength:", strength)

Positive probability: 0.99
Category: positive
Strength: 2


In [21]:
import os, requests
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


API_URL = "https://jev-ai.org/api/v1/systemone/"
MODEL   = "jev-1.13"

QUESTIONS = {
    "recommend": {
        "type": "noul",
        "instructions": "Would the reviewer recommend this movie to others?",
    },
    "opinion": {
        "type": "choice",
        "instructions": "What is the reviewer's overall opinion of the movie?",
        "criteria": {
            "positive": "The reviewer clearly likes the movie.",
            "negative": "The reviewer clearly dislikes the movie.",
            "mixed": "The reviewer has both positive and negative opinions.",
        },
    },
    "sentiment": {
        "type": "score",
        "instructions": "How positive is the reviewer's overall sentiment?",
        "criteria": [
            "Very negative",
            "Somewhat negative",
            "Neutral or mixed",
            "Somewhat positive",
            "Very positive",
        ],
    },
}

def call_jev(text):
    r = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={"model": MODEL, "state": text, "questions": QUESTIONS},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

def bar(label, pct, color):
    return f"""
    <div style="margin:6px 0">
      <div style="display:flex;justify-content:space-between;font-size:13px">
        <span>{label}</span><span>{pct:.0f}%</span></div>
      <div style="background:#e5e7eb;border-radius:6px;height:10px">
        <div style="width:{pct}%;background:{color};height:10px;border-radius:6px"></div>
      </div>
    </div>"""

def render(result):
    a = result["answers"]

    rec = a["recommend"]["noul"] * 100

    op = a["opinion"]
    op_colors = {"positive": "#16a34a", "negative": "#dc2626", "mixed": "#d97706"}
    op_bars = "".join(
        bar(k, v * 100, op_colors.get(k, "#6366f1"))
        for k, v in sorted(op["probabilities"].items(), key=lambda kv: -kv[1])
    )

    s = a["sentiment"]
    legend = s["legend"]
    s_label = legend[str(round(s["score"]))]
    s_bars = "".join(
        bar(legend[k], v * 100, "#6366f1") for k, v in s["probabilities"].items()
    )

    latency = result.get("latency_ms", "n/a")

    return f"""
    <div style="font-family:sans-serif;background:#fff;color:#111;border:1px solid #e5e7eb;
                border-radius:12px;padding:16px;max-width:560px">
      <h4 style="margin:0 0 8px">1. Would they recommend it? <small>(Noul)</small></h4>
      {bar("probability of yes", rec, "#16a34a")}

      <h4 style="margin:16px 0 4px">2. Overall opinion <small>(Choice)</small></h4>
      <div style="font-size:15px">Pick: <b>{op['choice']}</b>
         &nbsp;(confidence {op['confidence']*100:.0f}%)</div>
      {op_bars}

      <h4 style="margin:16px 0 4px">3. Sentiment strength <small>(Score)</small></h4>
      <div style="font-size:15px">Score: <b>{s['score']:.2f} / 4</b> &mdash; {s_label}</div>
      {s_bars}

      <div style="margin-top:12px;font-size:12px;color:#6b7280">
        Round-trip latency reported by API: {latency} ms
      </div>
    </div>"""

examples = {
    "Pick an example...": "",
    "Very positive": "Absolutely loved it! Brilliant acting and a story that kept me hooked till the end.",
    "Mixed": "The visuals were stunning and the lead actor was great, but the plot dragged and the ending felt rushed.",
    "Negative": "Boring, predictable, and way too long. I almost walked out halfway through.",
    "Sarcastic (tricky)": "Oh great, another sequel nobody asked for. Two hours of my life I'll never get back.",
}

dropdown = widgets.Dropdown(options=list(examples.keys()), layout=widgets.Layout(width="320px"))
textbox = widgets.Textarea(
    placeholder="Type or paste a movie review here...",
    layout=widgets.Layout(width="560px", height="110px"),
)
button = widgets.Button(description="Analyze with Jev", button_style="primary",
                        layout=widgets.Layout(width="180px"))
out = widgets.Output()

def on_pick(change):
    if change["new"] != "Pick an example...":
        textbox.value = examples[change["new"]]

def on_click(_):
    text = textbox.value.strip()
    with out:
        clear_output()
        if not text:
            print("Please enter a review first.")
            return
        print("Asking Jev...")
        try:
            result = call_jev(text)
            clear_output()
            display(HTML(render(result)))
        except requests.HTTPError as e:
            clear_output()
            print(f"API error {e.response.status_code}: {e.response.text[:300]}")
        except Exception as e:
            clear_output()
            print("Something went wrong:", e)

dropdown.observe(on_pick, names="value")
button.on_click(on_click)

display(HTML("<h3 style='font-family:sans-serif'>Movie Review Analyzer (powered by Jev)</h3>"))
display(dropdown, textbox, button, out)

Dropdown(layout=Layout(width='320px'), options=('Pick an example...', 'Very positive', 'Mixed', 'Negative', 'S…

Textarea(value='', layout=Layout(height='110px', width='560px'), placeholder='Type or paste a movie review her…

Button(button_style='primary', description='Analyze with Jev', layout=Layout(width='180px'), style=ButtonStyle…

Output()